In [14]:
!wget -O sales.csv https://raw.githubusercontent.com/necromancersanju-netizen/data-mining-data-lake-assignment/main/dataset/sales.csv

--2026-10-01 13:13:05--  https://raw.githubusercontent.com/necromancersanju-netizen/data-mining-data-lake-assignment/main/dataset/sales.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 109537 (107K) [text/plain]
Saving to: ‘sales.csv’

sales.csv           100%[===================>] 106.97K  --.-KB/s    in 0.01s   

2026-10-01 13:13:05 (7.34 MB/s) - ‘sales.csv’ saved [109537/109537]



In [2]:
!pip install -q pyspark

In [3]:
import os
import shutil

os.makedirs("data_lake/bronze", exist_ok=True)
os.makedirs("data_lake/silver", exist_ok=True)

shutil.copy("sales.csv", "data_lake/bronze/sales.csv")

print("Bronze file created successfully")

Bronze file created successfully


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("SalesDataLake") \
    .getOrCreate()

df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("data_lake/bronze/sales.csv")

df.printSchema()
df.show(10, truncate=False)

print("Total records:", df.count())
print("Columns:", df.columns)

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name  |product              |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+---------------+---------------------+---------+--------+----------+--------

In [5]:
print("=== DUPLICATES ===")
print("Duplicate rows:", df.count() - df.dropDuplicates().count())

print("\n=== NULL VALUES ===")
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

print("\n=== DATA TYPES ===")
df.printSchema()

=== DUPLICATES ===
Duplicate rows: 10

=== NULL VALUES ===
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|       0|          0|           10|      0|       0|       0|         0|               0|             8|   8|    0|         0|           0|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+


=== DATA TYPES ===
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |

In [6]:
print("=== TEXT VALUES ===")

for c in ["customer_name", "product", "category", "payment_method", "city", "state", "order_status"]:
    print(f"\n{c}:")
    df.select(c).distinct().show(30, truncate=False)

print("\n=== NUMERICAL RANGES ===")

df.select(
    min("quantity").alias("min_quantity"),
    max("quantity").alias("max_quantity"),
    min("unit_price").alias("min_unit_price"),
    max("unit_price").alias("max_unit_price"),
    min("discount_percent").alias("min_discount"),
    max("discount_percent").alias("max_discount")
).show()

print("\n=== ORDER DATE VALUES ===")

df.select("order_date").distinct().show(30, truncate=False)

=== TEXT VALUES ===

customer_name:
+------------------+
|customer_name     |
+------------------+
|Pranav Jain       |
|Tanvi Rao         |
|Meera Pandey      |
|Tanvi Patel       |
|Rahul Jain        |
|Kavya Bose        |
|Siddharth Menon   |
|Divya Naidu       |
|Aditi Agarwal     |
|Aman Naidu        |
|Siddharth Malhotra|
|Sneha Chopra      |
|Vikram Reddy      |
|Harsh Iyer        |
|Tanvi Naidu       |
|Swathi Gupta      |
|Shreya Gupta      |
|Harsh Pillai      |
|Pooja Jain        |
|Siddharth Kapoor  |
|Keerthi Tripathi  |
|Aditya Chopra     |
|Arjun Agarwal     |
|Vivaan Nair       |
|Deepak Pandey     |
|Pooja Verma       |
|Neha Sharma       |
|Divya Chopra      |
|Aarav Nair        |
|Kavya Malhotra    |
+------------------+
only showing top 30 rows

product:
+----------------------+
|product               |
+----------------------+
|Power Bank            |
|Wireless Mouse        |
|Dumbbell Set          |
|Mixer Grinder         |
|Jacket                |
|USB-C Hub     

In [9]:
from pyspark.sql.functions import *

silver = df.dropDuplicates()

for c, dtype in silver.dtypes:
    if dtype == "string":
        silver = silver.withColumn(c, trim(col(c)))

text_columns = [
    "customer_name",
    "product",
    "category",
    "payment_method",
    "city",
    "state",
    "order_status"
]

for c in text_columns:
    silver = silver.withColumn(c, initcap(col(c)))

silver = silver.fillna({
    "customer_name": "Unknown",
    "payment_method": "Unknown",
    "city": "Unknown"
})

silver = silver.withColumn(
    "quantity",
    when(col("quantity") > 0, col("quantity")).otherwise(None)
)

silver = silver.withColumn(
    "unit_price",
    when(col("unit_price") >= 0, col("unit_price")).otherwise(None)
)

silver = silver.withColumn(
    "discount_percent",
    when(
        (col("discount_percent") >= 0) &
        (col("discount_percent") <= 100),
        col("discount_percent")
    ).otherwise(None)
)

for c in ["quantity", "unit_price", "discount_percent"]:
    median_value = silver.approxQuantile(c, [0.5], 0.01)[0]
    silver = silver.fillna({c: median_value})

silver = silver.withColumn(
    "order_date_clean",
    regexp_replace(col("order_date"), "/", "-")
)

silver = silver.withColumn(
    "order_date",
    expr("try_cast(order_date_clean AS DATE)")
).drop("order_date_clean")

silver = silver.filter(col("order_date").isNotNull())

print("Rows before preprocessing:", df.count())
print("Rows after preprocessing:", silver.count())

silver.show(10, truncate=False)
silver.printSchema()

Rows before preprocessing: 1000
Rows after preprocessing: 985
+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name     |product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100331  |C0005      |Sakshi Joshi      |Football              |Sports         |2       |670       |30              |Credit Card     |Bengaluru |Karnataka     |2025-06-06|Cancelled   |
|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books          |4       |670       |0               |Net Banking     |Pune      |Maharashtra   |2025-05-08|Delivered   |
|100506  |C02

In [10]:
import glob
import shutil
import os

silver_temp = "data_lake/silver/temp"

silver.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(silver_temp)

part_file = glob.glob(silver_temp + "/part-*.csv")[0]

shutil.copy(part_file, "data_lake/silver/sales_silver.csv")

print("Silver dataset saved successfully.")
print("Location: data_lake/silver/sales_silver.csv")

Silver dataset saved successfully.
Location: data_lake/silver/sales_silver.csv


In [11]:
silver_check = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("data_lake/silver/sales_silver.csv")

print("Silver row count:", silver_check.count())

silver_check.show(10, truncate=False)
silver_check.printSchema()

Silver row count: 985
+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name     |product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100331  |C0005      |Sakshi Joshi      |Football              |Sports         |2       |670       |30              |Credit Card     |Bengaluru |Karnataka     |2025-06-06|Cancelled   |
|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books          |4       |670       |0               |Net Banking     |Pune      |Maharashtra   |2025-05-08|Delivered   |
|100506  |C0228      |Diya Tripathi     |Competitive 

In [12]:
import os

print("===== FINAL VERIFICATION =====")
print("Bronze row count:", df.count())
print("Silver row count:", silver_check.count())
print("Silver file exists:", os.path.exists("data_lake/silver/sales_silver.csv"))

===== FINAL VERIFICATION =====
Bronze row count: 1000
Silver row count: 985
Silver file exists: True


## Preprocessing Summary

The following preprocessing operations were performed:

1. Exact duplicate rows were removed.
2. Missing values in text fields were handled using suitable values.
3. Leading and trailing spaces were removed from text fields.
4. Text capitalization was standardized.
5. Invalid quantity values were handled.
6. Invalid unit price values were handled.
7. Invalid discount percentages were handled.
8. Dates were converted to a consistent `YYYY-MM-DD` format.
9. Clearly invalid dates were removed.
10. The processed data was saved as the Silver layer in CSV format.

### Record Count

- Records before preprocessing: 1000
- Records after preprocessing: 985

### Data Lake Structure

```text
data_lake/
├── bronze/
│   └── sales.csv
└── silver/
    └── sales_silver.csv
    

In [13]:
from google.colab import files

files.download("data_lake/silver/sales_silver.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>